# 📊 Fabric Racing Game - Live Dashboard

Real-time analytics for race telemetry using KQL.

**Visualizations:**
- Live positions on track
- Score & multiplier trends
- Level completion times
- Event heatmap

In [ ]:
# ⚙️ SETUP — nessuna configurazione manuale
# Trova da solo il "Query URI" del database RaceData e usa le REST API (nessun pacchetto da installare).
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output
import time

DATABASE_NAME = "RaceData"

def _tok(res):
    try:
        import notebookutils
        return notebookutils.credentials.getToken(res)
    except Exception:
        import subprocess
        return subprocess.run(f"az account get-access-token --resource {res} --query accessToken -o tsv",
                              capture_output=True, text=True, shell=True).stdout.strip()

# Auto-rileva il Query URI dell'Eventhouse per RaceData
KUSTO_CLUSTER = None
try:
    import notebookutils
    _ws = notebookutils.runtime.context.get("currentWorkspaceId")
    _H = {"Authorization": f"Bearer {_tok('https://api.fabric.microsoft.com')}"}
    _dbs = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{_ws}/items?type=KQLDatabase",
                        headers=_H).json().get("value", [])
    _db = next((d for d in _dbs if d["displayName"] == DATABASE_NAME), None)
    if _db:
        KUSTO_CLUSTER = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{_ws}/kqlDatabases/{_db['id']}",
            headers=_H).json()["properties"]["queryServiceUri"]
except Exception as _e:
    print("Auto-detect non riuscito:", _e)

# Fallback manuale (solo se l'auto-detect fallisce):
#   apri il database RaceData -> nella barra dei dettagli in alto copia "Query URI"
#   (finisce con .kusto.fabric.microsoft.com). NON usare l'Ingestion URI.
if not KUSTO_CLUSTER:
    KUSTO_CLUSTER = "https://trd-XXXXX.kusto.fabric.microsoft.com"

print("Query URI:", KUSTO_CLUSTER)

def run_query(query: str) -> pd.DataFrame:
    """Esegue una query KQL via REST e restituisce un DataFrame."""
    tok = _tok("https://kusto.kusto.windows.net")
    r = requests.post(f"{KUSTO_CLUSTER}/v2/rest/query",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"db": DATABASE_NAME, "csl": query}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"KQL HTTP {r.status_code}: {r.text[:400]}")
    for frame in r.json():
        if frame.get("FrameType") == "DataTable" and frame.get("TableKind") == "PrimaryResult":
            cols = [c["ColumnName"] for c in frame["Columns"]]
            return pd.DataFrame(frame["Rows"], columns=cols)
    return pd.DataFrame()

print("✅ Dashboard pronta. Esegui le celle sotto.")


## 🏁 Latest Race Session

In [ ]:
# Get latest session
latest_session_query = """
GameEvents
| summarize LastEvent = max(todatetime(Timestamp)) by SessionId
| top 1 by LastEvent desc
| project SessionId
"""
session_df = run_query(latest_session_query)
SESSION_ID = session_df['SessionId'].iloc[0] if len(session_df) > 0 else None
print(f"📍 Active Session: {SESSION_ID}")

## 📈 Events Summary

In [ ]:
events_query = f"""
GameEvents
| where SessionId == "{SESSION_ID}"
| summarize Count = count() by EventType
| order by Count desc
"""
events_df = run_query(events_query)
fig = px.bar(events_df, x='EventType', y='Count', 
             title='Events by Type',
             color='EventType')
fig.show()

## 🏁 Score Progress Over Time

In [ ]:
progress_query = f"""
GameEvents
| where SessionId == "{SESSION_ID}"
| where isnotempty(Score)
| project Timestamp = todatetime(Timestamp), PlayerId, Level, Score=tolong(Score)
| order by Timestamp asc
"""
progress_df = run_query(progress_query)

if len(progress_df) > 0:
    fig = px.line(progress_df, x='Timestamp', y='Score',
                  color='PlayerId', markers=True,
                  title='🏁 Race Progress — Score over Time')
    fig.update_layout(xaxis_title='Time', yaxis_title='Score')
    fig.show()
else:
    print("No score data yet for this session")


## ⏱️ Level Completion Times

In [ ]:
level_times_query = f"""
GameEvents
| where SessionId == "{SESSION_ID}"
| where EventType in ("LevelStart", "LevelComplete")
| summarize StartT = minif(todatetime(Timestamp), EventType == "LevelStart"),
            EndT   = maxif(todatetime(Timestamp), EventType == "LevelComplete")
        by Level, LevelName
| where isnotnull(StartT) and isnotnull(EndT)
| extend LevelTimeSeconds = datetime_diff('millisecond', EndT, StartT) / 1000.0
| project Level, LevelName, LevelTimeSeconds
| order by Level asc
"""
lap_df = run_query(level_times_query)

if len(lap_df) > 0:
    fig = px.bar(lap_df, x='LevelName', y='LevelTimeSeconds',
                 title='Level Completion Times')
    fig.show()
else:
    print("No level completion data yet")

## 🔥 Score Multiplier Over Time

In [ ]:
multiplier_query = f"""
GameEvents
| where SessionId == "{SESSION_ID}"
| where EventType == "StarCollected"
| project Timestamp = todatetime(Timestamp), Multiplier = toint(Multiplier)
| order by Timestamp asc
"""
speed_df = run_query(multiplier_query)

if len(speed_df) > 0:
    fig = px.line(speed_df, x='Timestamp', y='Multiplier',
                  markers=True,
                  title='Score Multiplier Over Time')
    fig.show()
else:
    print("No multiplier data yet")

## 🔥 Events by Type & Level

In [ ]:
heatmap_query = f"""
GameEvents
| where SessionId == "{SESSION_ID}"
| summarize EventCount = count() by EventType, Level
"""
heatmap_df = run_query(heatmap_query)

if len(heatmap_df) > 0:
    # Pivot for heatmap
    pivot_df = heatmap_df.pivot(index='EventType', columns='Level', values='EventCount').fillna(0)

    fig = px.imshow(pivot_df,
                    title='Events by Type & Level',
                    labels=dict(x='Level', y='Event Type', color='Count'),
                    color_continuous_scale='RdYlGn')
    fig.show()
else:
    print("No event data yet")

## 🔄 Auto-Refresh Dashboard

Run this cell to continuously refresh the dashboard during a live race:

In [ ]:
# Uncomment to enable auto-refresh (Ctrl+C to stop)
# while True:
#     clear_output(wait=True)
#     # Re-run position query and display
#     positions_df = run_query(positions_query)
#     display(positions_df)
#     time.sleep(2)